# Qwen2.5-7B on vLLM: BF16 vs FP8 vs AWQ vs GPTQ
Runtime → Change runtime type → **L4 GPU**, then Run all.

- Installs vLLM 0.32.0 and downloads three models, about 27 GB (about 15 minutes).
- Runs a smoke test (about 10 minutes), then the full run over 5 configs (about 75 minutes).
- Roughly 1.75 hours in total, about 4 compute units.
- At the end it downloads `quant_results.zip`.

The notebook **stops** if vLLM fails to install or the smoke test fails. In the full run, a config that fails to start is skipped and listed at the end. If anything fails, send the output of that cell.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

In [ ]:
import os
if not os.path.isdir('/content/lowbit-decode-gpu'):
    !git clone -q https://github.com/FredWen-S/lowbit-decode-gpu.git /content/lowbit-decode-gpu
%cd /content/lowbit-decode-gpu
!git pull -q && git log -1 --format='%h %s'

vLLM in its own virtualenv, pinned to the version of the serving runs.

In [ ]:
!pip -q install uv
!uv venv --clear --python 3.12 /content/venv-vllm
!uv pip install --python /content/venv-vllm/bin/python vllm==0.32.0 ninja 2>&1 | tail -25

In [ ]:
import os, subprocess
r = subprocess.run(['/content/venv-vllm/bin/python', '-c', 'import vllm; print(vllm.__version__)'], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-2000:])
assert r.returncode == 0, 'vllm did not install; see the install cell above'
assert os.path.exists('/content/venv-vllm/bin/ninja'), 'ninja missing in venv-vllm'

In [ ]:
from huggingface_hub import snapshot_download
for repo in ['Qwen/Qwen2.5-7B-Instruct', 'Qwen/Qwen2.5-7B-Instruct-AWQ', 'Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4']:
    path = snapshot_download(repo, local_dir='/content/models/' + repo.split('/')[1])
    print(path)
!pip -q install aiohttp datasets
!du -sh /content/models/*

Smoke test: BF16 and AWQ, few requests, 100 MMLU items, 10 WikiText chunks.

In [ ]:
!rm -f /content/SMOKE_FAILED
!python bench/quant_sweep.py --models-dir /content/models --python /content/venv-vllm/bin/python --quick --configs bf16 awq --out results/quant_quick || touch /content/SMOKE_FAILED
import os
assert not os.path.exists('/content/SMOKE_FAILED'), 'smoke test failed; see the output above'

Full run: all 5 configs.

In [ ]:
!python bench/quant_sweep.py --models-dir /content/models --python /content/venv-vllm/bin/python --out results/quant

In [ ]:
!zip -qr /content/quant_results.zip results/quant results/quant_quick
from google.colab import files
files.download('/content/quant_results.zip')